### Transfer Learning Tutorial 

Our task is to separate cats, dogs, and birds.
Claude code helped with the preprocessing, some setup, and the test code / confusion matrix.

In [ ]:
import torch
import torchvision

# information for creating the demo 

# pip install --upgrade gradio
# import gradio as gr

# def greet(name, itensity):
#   return "Hello, " + name + "!" * int(intensity)

# demo = gr.Interface(
#    fn = greet,
#    inputs = ["text", "slider"],
#    outputs = ["text"],
#    api_name = "predict"
# )

# TODO: After you run the code to see what's going on, 
# comment out the following 2 lines. 
vgg_model = torchvision.models.vgg16() 
print(vgg_model.features[0].bias) # type: ignore

# Instead use this one, which loads the pre-trained weights.  
vgg_model = torchvision.models.vgg16(weights=torchvision.models.VGG16_Weights.IMAGENET1K_V1)
print(vgg_model.features[0].bias) # type: ignore

# This standardization of R,G,B is specific to VGG. Notice it expects square images of size 256x256. 
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize(256),
    torchvision.transforms.CenterCrop(224),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.],
       requires_grad=True)
Parameter containing:
tensor([ 0.4034,  0.3778,  0.4644, -0.3228,  0.3940, -0.3953,  0.3951, -0.5496,
         0.2693, -0.7602, -0.3508,  0.2334, -1.3239, -0.1694,  0.3938, -0.1026,
         0.0460, -0.6995,  0.1549,  0.5628,  0.3011,  0.3425,  0.1073,  0.4651,
         0.1295,  0.0788, -0.0492, -0.5638,  0.1465, -0.3890, -0.0715,  0.0649,
         0.2768,  0.3279,  0.5682, -1.2640, -0.8368, -0.9485,  0.1358,  0.2727,
         0.1841, -0.5325,  0.3507, -0.0827, -1.0248, -0.6912, -0.7711,  0.2612,
         0.4033, -0.4802, -0.3066,  0.5807, -1.3325,  0.4844, -0.8160,  0.2386,
         0.2300,  0.4979,  0.5553,  0.5230, -0.2182,  0.0117, -0.5516,  0.2108],
       

In [30]:
import numpy as np
from datasets import load_dataset
from torchvision import transforms
 
LABEL_COL = 'style'
N = 3000   # ~1.8 GB as float32, same as your original note
 
ds = load_dataset("huggan/wikiart", split="train")
class_names = ds.features[LABEL_COL].names

 
def load_split(split_ds):
    """Returns X (N, 3, 224, 224) float32 and y (N,) int64."""
    X = np.empty((len(split_ds), 3, 224, 224), dtype=np.float32)
    for i, example in enumerate(split_ds):
        X[i] = preprocess(example["image"].convert("RGB")).numpy()
    y = np.array(split_ds[LABEL_COL], dtype=np.int64)   # reads only the label column
    return X, y
 
# Random subset, then carve out validation/test (train is the only split on the Hub)
subset = ds.shuffle(seed=4).select(range(N))
tmp = subset.train_test_split(test_size=0.2, seed=42)
val_test = tmp["test"].train_test_split(test_size=0.5, seed=42)
 
X_train, Y_train = load_split(tmp["train"])
X_val,   Y_val   = load_split(val_test["train"])
X_test,  Y_test  = load_split(val_test["test"])
 
for name, X, Y in [('train', X_train, Y_train), ('validation', X_val, Y_val), ('test', X_test, Y_test)]:
    print(name, X.shape, Y.shape, 'class counts:', np.bincount(Y, minlength=len(class_names)))

class_names = ds.features[LABEL_COL].names


Resolving data files:   0%|          | 0/72 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/45 [00:00<?, ?it/s]

train (2400, 3, 224, 224) (2400,) class counts: [ 86   4   4 144 119  54  22  69  41 219  27  50 372  41  31  84  11  74
  17  47 181 285  66 194 121   5  32]
validation (300, 3, 224, 224) (300,) class counts: [12  1  0 16 14  1  2  9  7 24  3  3 50  4  2  9  1  6  3  7 29 38 12 23
 19  1  4]
test (300, 3, 224, 224) (300,) class counts: [15  1  1 14 13  5  2 10  6 25  4  4 46  7  5  5  3  9  2  3 26 48  9 23
 11  0  3]


In [31]:
# Start by printing the model. Expand it to see the whole thing. Note that it has 3 parts: features, avgpool, and classifier.
# We'll do the simple thing, which is to remove the classifier.
print(vgg_model)


VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1

In [32]:
# This is a simple 3 class problem (bird, cat, dog) 
# so we'll see if a single Linear layer works. We rebuild it.
class AnimalModel(torch.nn.Module):
    def __init__(self, vgg_model):
        super().__init__()
        self.features = vgg_model.features
        self.avgpool = vgg_model.avgpool
        self.classifier = torch.nn.Sequential(torch.nn.Linear(25088, 30), torch.nn.Linear(30, 27))

    def forward(self, x):
        f = self.features(x)
        a = self.avgpool(f)
        flat = torch.flatten(a, 1)
        zE = self.classifier(flat)
        return zE 
    
animal_model = AnimalModel(vgg_model)
print(animal_model)

AnimalModel(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): MaxPool2d(kernel_size=2, stride=2, padding=0, di

In [33]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [34]:
import time
import copy
import plotly.graph_objects as go
from torch.utils.data import TensorDataset, DataLoader

# Will give the same random weights each time we run the code, 
# which is helpful for debugging and reproducibility
torch.manual_seed(42) 
model = AnimalModel(copy.deepcopy(vgg_model)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.features.parameters():
    param.requires_grad = False

# Hyperparameters
epsilon = 0.0001 # Learning rate.
batch_size = 32

# Shuffle the training data every epoch so batches differ from epoch to epoch
train_dataset = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(Y_train))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Validation and test sets are too big to push through VGG in one go, so evaluate in batches
val_loader = DataLoader(TensorDataset(torch.from_numpy(X_val), torch.from_numpy(Y_val)), batch_size=batch_size)
test_loader = DataLoader(TensorDataset(torch.from_numpy(X_test), torch.from_numpy(Y_test)), batch_size=batch_size)

# Use CCE loss. Adam (adaptive momentum) is a fancy version of gradient descent 
criterion = torch.nn.CrossEntropyLoss() 
# Only pass the unfrozen parameters to the optimizer
optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=epsilon)


In [35]:
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

# # Optional: break it down by layer
# for name, p in model.named_parameters():
#     print(name, tuple(p.shape), p.numel())

Total trainable parameters: 753507


In [36]:
def train_model(model, optimizer, num_epochs, checkpoint_path):
    """Train, keep the checkpoint with the lowest validation loss, and plot the losses."""
    training_losses = []
    validation_losses = []
    best_validation_loss = float('inf')
    best_validation_epoch = -1
    training_start = time.time()

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0
        num_batches = 0
        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)
            optimizer.zero_grad() 
            loss = criterion(outputs, labels)
            loss.backward() 
            optimizer.step()
            
            epoch_loss += loss.item()
            num_batches += 1

        avg_loss = epoch_loss / num_batches
        training_losses.append(avg_loss)

        # Compute validation loss. Don't need gradients here.
        model.eval()
        with torch.no_grad():
            val_total = 0
            for images, labels in val_loader:
                val_outputs = model(images.to(device))
                val_total += criterion(val_outputs, labels.to(device)).item() * len(labels)
            validation_loss = val_total / len(X_val)
            validation_losses.append(validation_loss)

        # Save the model if this is the best validation loss so far
        if validation_loss < best_validation_loss:
            best_validation_loss = validation_loss
            best_validation_epoch = epoch
            torch.save(model.state_dict(), checkpoint_path)
            best_marker = ' *'
        else:
            best_marker = ''

        print(f'Epoch [{epoch}/{num_epochs}], Train Loss: {avg_loss:.5f}, Val Loss: {validation_loss:.5f}{best_marker}')

    total_time = time.time() - training_start
    print(f'Average time per epoch: {total_time / num_epochs:.2f}s')
    print(f'Best validation loss: {best_validation_loss:.5f} (epoch {best_validation_epoch})')

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=list(range(len(training_losses))), y=training_losses, mode='lines', name='Training Loss'))
    fig.add_trace(go.Scatter(x=list(range(len(validation_losses))), y=validation_losses, mode='lines', name='Validation Loss'))
    fig.update_layout(title="Training and Validation Loss Over Time", xaxis_title="Epoch", yaxis_title="Loss")
    fig.show()

# Transfer learning: pretrained VGG features (frozen) + new Linear classifier
train_model(model, optimizer, num_epochs=10, checkpoint_path='animal_classifier.pth')

Epoch [0/10], Train Loss: 2.66476, Val Loss: 2.33366 *
Epoch [1/10], Train Loss: 1.50032, Val Loss: 2.20371 *
Epoch [2/10], Train Loss: 0.95115, Val Loss: 2.14916 *
Epoch [3/10], Train Loss: 0.63484, Val Loss: 2.15604
Epoch [4/10], Train Loss: 0.43635, Val Loss: 2.15983
Epoch [5/10], Train Loss: 0.31301, Val Loss: 2.17389
Epoch [6/10], Train Loss: 0.23314, Val Loss: 2.17683
Epoch [7/10], Train Loss: 0.17742, Val Loss: 2.20020
Epoch [8/10], Train Loss: 0.13937, Val Loss: 2.22246
Epoch [9/10], Train Loss: 0.11167, Val Loss: 2.23709
Average time per epoch: 23.02s
Best validation loss: 2.14916 (epoch 2)


In [37]:
def evaluate_model(model, checkpoint_path):
    """Load the best checkpoint (lowest validation loss) and report test accuracy and confusion matrix."""
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    model.eval()

    predictions = []
    with torch.no_grad():
        for images, labels in test_loader:
            predictions.append(model(images.to(device)).argmax(dim=1).cpu())
    predictions = torch.cat(predictions).numpy()


    print(f'Test accuracy: {(predictions == Y_test).mean():.4f}')

    # Confusion matrix: rows are the true class, columns are the predicted class
    confusion = np.zeros((27, 27), dtype=int)
    for true, pred in zip(Y_test, predictions):
        confusion[true, pred] += 1
    print('Confusion matrix (rows = true, cols = predicted):', class_names)
    print(confusion)

evaluate_model(model, 'animal_classifier.pth')


Test accuracy: 0.3933
Confusion matrix (rows = true, cols = predicted): ['Abstract_Expressionism', 'Action_painting', 'Analytical_Cubism', 'Art_Nouveau', 'Baroque', 'Color_Field_Painting', 'Contemporary_Realism', 'Cubism', 'Early_Renaissance', 'Expressionism', 'Fauvism', 'High_Renaissance', 'Impressionism', 'Mannerism_Late_Renaissance', 'Minimalism', 'Naive_Art_Primitivism', 'New_Realism', 'Northern_Renaissance', 'Pointillism', 'Pop_Art', 'Post_Impressionism', 'Realism', 'Rococo', 'Romanticism', 'Symbolism', 'Synthetic_Cubism', 'Ukiyo_e']
[[ 2  0  0  0  0  1  0  0  0  1  0  0  5  0  0  0  0  0  0  1  2  0  0  1
   2  0  0]
 [ 0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1
   0  0  0]
 [ 0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0
   0  0  0]
 [ 0  0  0  3  0  0  0  0  0  2  0  0  0  0  0  0  0  0  0  1  1  4  0  1
   2  0  0]
 [ 0  0  0  0  5  0  0  0  0  0  0  0  3  0  0  0  0  1  0  0  0  3  0  1
   0  0  0]
 [ 0  0  0  0  0  3  0  0

In [38]:
# Baseline: the same architecture and training, but VGG16 starts with RANDOM weights
# (the frozen features are just a random projection). The gap between this and the
# pretrained result is what transfer learning buys us.
# torch.manual_seed(42)
# baseline_model = AnimalModel(torchvision.models.vgg16()).to(device)  # weights=None by default
# for param in baseline_model.features.parameters():
#     param.requires_grad = False
# baseline_optimizer = torch.optim.Adam([p for p in baseline_model.parameters() if p.requires_grad], lr=epsilon)

# train_model(baseline_model, baseline_optimizer, num_epochs=10, checkpoint_path='animal_baseline.pth')
# evaluate_model(baseline_model, 'animal_baseline.pth')


In [39]:
# finetune_model = AnimalModel(copy.deepcopy(vgg_model)).to(device)
# finetune_model.load_state_dict(torch.load('animal_classifier.pth', map_location=device))

# for param in finetune_model.features.parameters():
#     param.requires_grad = True

# # Cool trick: we can use different learning rate on different params, if we pass a list of param/lr pairs.
# # This lets us train the feature layers more slowly.  
# finetune_optimizer = torch.optim.Adam([
#     {'params': finetune_model.features.parameters(), 'lr': 1e-5},
#     {'params': finetune_model.classifier.parameters(), 'lr': epsilon},
# ])

# train_model(finetune_model, finetune_optimizer, num_epochs=5, checkpoint_path='animal_finetuned.pth')
# evaluate_model(finetune_model, 'animal_finetuned.pth')


In [40]:
# Here is what feature extraction looks like:
@torch.no_grad()
def extract_features(model, X, batch_size=32):
    """Run the frozen VGG body once and return the flattened 25088-dim features."""
    model.eval()
    out = []
    for i in range(0, len(X), batch_size):
        x = torch.from_numpy(X[i:i + batch_size]).to(device)
        out.append(torch.flatten(model.avgpool(model.features(x)), 1).cpu())
    return torch.cat(out)

# Uncomment this to extract features. It might take a minute, but you only need to do it once.
# F_train = extract_features(model, X_train)   # (2100, 25088), about 210 MB
# F_val   = extract_features(model, X_val)
# F_test  = extract_features(model, X_test)

# This is the setup needed to train just the Linear layer on the cached features. Takes seconds.
# head = torch.nn.Linear(25088, 3).to(device)
# head_opt = torch.optim.Adam(head.parameters(), lr=epsilon)
# head_loader = DataLoader(TensorDataset(F_train, torch.from_numpy(Y_train)),
#                          batch_size=batch_size, shuffle=True)
# ... need training loop